In [1]:
"""
mvc.py
======
Módulo de cálculo de composições de máximo valor (MVC – Maximum Value
Composite) para índices espectrais derivados do satélite GOES-16.

Dois tipos de MVC são gerados:
    - MVC diário  : máximo de todos os horários disponíveis em um dia.
    - MVC horário : máximo de todas as cenas disponíveis dentro de uma hora.

Estrutura de entrada esperada (gerada por spectral_index.py):
    <input_base>/<INDICE>/<ano>/<dia>/<hora_HH>/<horaHHMMSS>.tif

Estrutura de saída:
    MVC diário  → <output_base>/MVC_<INDICE>/<ano>/<dia>/MVC_<INDICE>_<ano><dia>.tif
    MVC horário → <output_base>/MVC_<INDICE>/<ano>/<dia>/<hora_HH>/MVC_<INDICE>_<ano><dia><hora_HH>.tif

Dependências:
    numpy, rasterio

Uso típico:
    for indice in INDICES:
        processar_mvc_indice(
            indice=indice,
            input_base='Arquivos',
            output_base='Arquivos',
        )
"""

import os
import warnings
from pathlib import Path
from typing import List, Optional

import numpy as np
import rasterio
from rasterio.warp import Resampling, reproject

# Fallback para tqdm
try:
    from tqdm import tqdm
except ImportError:
    def tqdm(iterable, desc="", **kwargs):
        print(f"{desc}..." if desc else "Processando...")
        return iterable

# ---------------------------------------------------------------------------
# Constantes do módulo
# ---------------------------------------------------------------------------

# Índices espectrais disponíveis — deve coincidir com INDICES em spectral_index.py
INDICES = ["NDVI", "NBR", "NBR2", "NDMI", "MIRBI", "EVI", "SAVI"]

# Valor sentinela para pixels sem dado em todos os rasters do grupo
NODATA_FALLBACK = np.nan


# ---------------------------------------------------------------------------
# Função central de cálculo do MVC
# ---------------------------------------------------------------------------

def calcular_mvc(file_list: List[str], output_path: str) -> None:
    """
    Calcula o MVC (máximo valor, ignorando NaN) de uma lista de rasters e
    salva o resultado como GeoTIFF.

    O primeiro arquivo define a grade de referência (CRS, transform,
    dimensões). Os demais são reprojetados para essa grade via
    Resampling.nearest antes do empilhamento.

    Parâmetros:
        file_list   (list[str]): Caminhos dos rasters de entrada (.tif).
        output_path (str):       Caminho completo do GeoTIFF de saída.

    Levanta:
        ValueError: Se `file_list` estiver vazia.
        FileNotFoundError: Se algum arquivo da lista não existir.
    """
    if not file_list:
        raise ValueError("Lista de arquivos vazia.")

    arrays = []
    ref_profile = None

    for i, file_path in enumerate(file_list):
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Arquivo não encontrado: '{file_path}'")

        with rasterio.open(file_path) as src:
            if i == 0:
                # Primeiro arquivo define a grade de referência
                ref_profile = src.profile.copy()
                arrays.append(src.read(1).astype(np.float32))
            else:
                # Reprojeta para a grade de referência antes de empilhar
                out_data = np.empty(
                    (ref_profile["height"], ref_profile["width"]),
                    dtype=np.float32,
                )
                reproject(
                    source=src.read(1).astype(np.float32),
                    destination=out_data,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=ref_profile["transform"],
                    dst_crs=ref_profile["crs"],
                    resampling=Resampling.nearest,
                )
                arrays.append(out_data)

    # Empilha e calcula máximo — warning de slice all-NaN escopado localmente
    stacked = np.stack(arrays, axis=0)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="All-NaN slice encountered")
        max_array = np.nanmax(stacked, axis=0)

    # Define nodata: usa o do perfil de referência ou NODATA_FALLBACK
    nodata = ref_profile.get("nodata") or NODATA_FALLBACK
    max_array = np.where(np.isnan(max_array), nodata, max_array)

    # Atualiza perfil de saída
    ref_profile.update({
        "dtype":   "float32",
        "count":   1,
        "nodata":  nodata,
        "compress": "lzw",
        "tiled":   True,
    })

    # Garante existência do diretório de saída (protegido contra path vazio)
    output_dir = os.path.dirname(output_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    with rasterio.open(output_path, "w", **ref_profile) as dst:
        dst.write(max_array.astype(np.float32), 1)


# ---------------------------------------------------------------------------
# Funções de orquestração por índice
# ---------------------------------------------------------------------------

def _coletar_tifs_recursivo(diretorio: Path) -> List[str]:
    """
    Coleta recursivamente todos os arquivos .tif dentro de um diretório.

    Parâmetros:
        diretorio (Path): Diretório raiz da busca.

    Retorna:
        list[str]: Caminhos ordenados dos arquivos encontrados.
    """
    return sorted(
        str(p) for p in diretorio.rglob("*.tif")
        if p.is_file()
    )


def _coletar_tifs_diretos(diretorio: Path) -> List[str]:
    """
    Coleta arquivos .tif diretamente em um diretório (sem recursão).

    Parâmetros:
        diretorio (Path): Diretório a inspecionar.

    Retorna:
        list[str]: Caminhos ordenados dos arquivos encontrados.
    """
    return sorted(
        str(p) for p in diretorio.glob("*.tif")
        if p.is_file()
    )


def processar_mvc_indice(
    indice: str,
    input_base: str,
    output_base: str,
    calcular_diario: bool = True,
    calcular_horario: bool = True,
) -> None:
    """
    Calcula MVCs diário e/ou horário para um índice espectral.

    Varre a árvore de diretórios do índice (gerada por spectral_index.py)
    e gera um MVC por dia e/ou por hora, conforme configurado.

    Parâmetros:
        indice           (str):  Nome do índice (ex.: 'NDVI').
        input_base       (str):  Diretório raiz onde os índices estão armazenados.
        output_base      (str):  Diretório raiz onde os MVCs serão salvos.
        calcular_diario  (bool): Se True, gera MVC diário (padrão: True).
        calcular_horario (bool): Se True, gera MVC horário (padrão: True).
    """
    indice_dir  = Path(input_base) / indice
    output_dir  = Path(output_base) / f"MVC_{indice}"

    if not indice_dir.exists():
        print(f"⚠️  Diretório do índice não encontrado: '{indice_dir}'")
        return

    anos = sorted(p for p in indice_dir.iterdir() if p.is_dir())
    if not anos:
        print(f"⚠️  Nenhum ano encontrado em '{indice_dir}'")
        return

    sucesso_diario  = 0
    sucesso_horario = 0
    erros: List[str] = []

    for ano_path in tqdm(anos, desc=f"MVC {indice}"):
        ano = ano_path.name
        dias = sorted(p for p in ano_path.iterdir() if p.is_dir())

        for dia_path in dias:
            dia = dia_path.name

            # ----------------------------------------------------------
            # MVC DIÁRIO: máximo de todos os horários do dia
            # ----------------------------------------------------------
            if calcular_diario:
                tif_files = _coletar_tifs_recursivo(dia_path)

                if tif_files:
                    out_path = output_dir / ano / dia / f"MVC_{indice}_{ano}{dia}.tif"
                    try:
                        calcular_mvc(tif_files, str(out_path))
                        sucesso_diario += 1
                    except Exception as e:
                        erros.append(f"Diário {ano}/{dia}: {e}")
                else:
                    print(f"  ⚠️  Sem .tif em {ano}/{dia} — MVC diário ignorado.")

            # ----------------------------------------------------------
            # MVC HORÁRIO: máximo das cenas dentro de cada hora
            # ----------------------------------------------------------
            if calcular_horario:
                horas = sorted(p for p in dia_path.iterdir() if p.is_dir())

                for hora_path in horas:
                    hora = hora_path.name
                    tif_files = _coletar_tifs_diretos(hora_path)

                    if tif_files:
                        out_path = (
                            output_dir / ano / dia / hora /
                            f"MVC_{indice}_{ano}{dia}{hora}.tif"
                        )
                        try:
                            calcular_mvc(tif_files, str(out_path))
                            sucesso_horario += 1
                        except Exception as e:
                            erros.append(f"Horário {ano}/{dia}/{hora}: {e}")
                    else:
                        print(f"  ⚠️  Sem .tif em {ano}/{dia}/{hora} — MVC horário ignorado.")

    # Resumo por índice
    print(f"\n{'─' * 50}")
    print(f"Índice  : {indice}")
    if calcular_diario:
        print(f"  ✅ MVCs diários  : {sucesso_diario}")
    if calcular_horario:
        print(f"  ✅ MVCs horários : {sucesso_horario}")
    if erros:
        print(f"  ❌ Erros ({len(erros)}):")
        for err in erros:
            print(f"     • {err}")
    print(f"{'─' * 50}\n")


# ---------------------------------------------------------------------------
# Execução
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    INPUT_BASE  = Path("Arquivos")
    OUTPUT_BASE = Path("Arquivos")

    for indice in INDICES:
        processar_mvc_indice(
            indice=indice,
            input_base=str(INPUT_BASE),
            output_base=str(OUTPUT_BASE),
            calcular_diario=True,
            calcular_horario=True,
        )

MVC NDVI: 100%|██████████| 1/1 [00:00<00:00,  3.70it/s]



──────────────────────────────────────────────────
Índice  : NDVI
  ✅ MVCs diários  : 2
  ✅ MVCs horários : 2
──────────────────────────────────────────────────



MVC NBR: 100%|██████████| 1/1 [00:00<00:00,  3.70it/s]



──────────────────────────────────────────────────
Índice  : NBR
  ✅ MVCs diários  : 2
  ✅ MVCs horários : 2
──────────────────────────────────────────────────



MVC NBR2: 100%|██████████| 1/1 [00:00<00:00,  4.00it/s]


...


──────────────────────────────────────────────────
Índice  : MIRBI
  ✅ MVCs diários  : 2
  ✅ MVCs horários : 2
──────────────────────────────────────────────────



MVC EVI: 100%|██████████| 1/1 [00:00<00:00,  4.68it/s]



──────────────────────────────────────────────────
Índice  : EVI
  ✅ MVCs diários  : 2
  ✅ MVCs horários : 2
──────────────────────────────────────────────────



MVC SAVI: 100%|██████████| 1/1 [00:00<00:00,  3.77it/s]


──────────────────────────────────────────────────
Índice  : SAVI
  ✅ MVCs diários  : 2
  ✅ MVCs horários : 2
──────────────────────────────────────────────────

